# Profiling => Measuring Performance With timeit & cProfile

Never guess what is slow. **Measure** first, then optimize the part that matters.

| Tool | Purpose |
|---|---|
| `time.perf_counter()` | High-resolution clock for a quick manual measurement |
| `timeit.timeit(stmt, number=n)` | Total time of `n` runs of a small snippet |
| `timeit.repeat(stmt, repeat=r, number=n)` | Several measurements. Use the **minimum** |
| `timeit.Timer(...)` | Reusable timer object |
| `%timeit` / `%%timeit` | Jupyter magic that picks the count for you |
| `python -m timeit "expr"` | Command-line version |
| `cProfile` | Shows **which functions** use the time |
| `pstats.Stats` | Sorts and prints profile results |
| `tracemalloc` | Tracks memory allocations |

---

## `time.perf_counter()`

```python
start = time.perf_counter()
work()
elapsed = time.perf_counter() - start
```

Good for measuring one long operation. One run can be noisy.

---

## `timeit`

`timeit` runs a small piece of code many times and disables garbage collection during the run.

```python
import timeit

timeit.timeit("sum(range(1000))", number=1000)
timeit.repeat("sum(range(1000))", repeat=5, number=1000)
```

### Important

* The result is the **total** time for `number` runs, in seconds.
* Take the **minimum** of several repeats. Higher values usually come from other things running on the machine, not from your code.
* A callable can be timed instead of a string: `timeit.timeit(func, number=1000)`.
* Setup code goes in `setup=` and is not timed.

---

## `cProfile`

Finds **where** a program spends its time.

```python
import cProfile, pstats

profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()

pstats.Stats(profiler).sort_stats("cumulative").print_stats(10)
```

| Column | Meaning |
|---|---|
| `ncalls` | Number of calls |
| `tottime` | Time inside the function itself |
| `cumtime` | Time including everything it calls |
| `percall` | Time per call |

* Sort by `cumulative` to find the slow **paths**.
* Sort by `tottime` to find the slow **functions**.
* Profiling adds overhead, so use it to compare functions, not to get exact times.

Terminal: `python -m cProfile -s cumulative script.py`.

---

## `tracemalloc` (Memory)

```python
import tracemalloc

tracemalloc.start()
work()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
```

`peak` is the highest memory use, in bytes, while tracing.

---

## Optimization Rules

1. Make it **correct** first.
2. **Measure** to find the real bottleneck.
3. Change **one thing** and measure again.
4. Prefer a better algorithm or data structure over micro-tweaks (for example, a `set` for membership tests).
5. Stop when it is fast enough.

## Source

https://docs.python.org/3/library/timeit.html

https://docs.python.org/3/library/profile.html

https://docs.python.org/3/library/tracemalloc.html

In [ ]:
import cProfile
import pstats
import time
import timeit
import tracemalloc

# time.perf_counter: one manual measurement
start = time.perf_counter()
sum(range(1_000_000))
elapsed = time.perf_counter() - start
print(elapsed >= 0)

# timeit: compare two ways to test membership
data_list = list(range(5000))
data_set = set(data_list)

list_time = min(timeit.repeat("4999 in data", globals={"data": data_list}, repeat=5, number=200))
set_time = min(timeit.repeat("4999 in data", globals={"data": data_set}, repeat=5, number=200))
print(f"list: {list_time:.5f}s   set: {set_time:.5f}s   set is faster: {set_time < list_time}")

# timeit with a callable
def build():
    result = []
    for n in range(1000):
        result.append(n * 2)
    return result

print(timeit.timeit(build, number=200) >= 0)

# cProfile: which functions use the time?
def square(i):
    return i * i

def slow_square_sum(n):
    total = 0
    for i in range(n):
        total += square(i)
    return total

profiler = cProfile.Profile()
profiler.enable()
slow_square_sum(20_000)
profiler.disable()

stats = pstats.Stats(profiler)
print(stats.total_calls > 20_000)                 # square() was called 20,000 times
names = set()
for func in stats.stats:                          # each key is (file, line, function name)
    names.add(func[2])
print("square" in names, "slow_square_sum" in names)

# tracemalloc: peak memory of an operation
tracemalloc.start()
numbers = list(range(100_000))
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(current > 0, peak >= current)

# Terminal versions:
#     python -m timeit "sum(range(1000))"
#     python -m cProfile -s cumulative script.py
# Jupyter:
#     %timeit sum(range(1000))